# Lab 8: Introducing Hyperparameter Tuning

Objectives:
- To gain hands-on experience tuning parameters
- To implement concepts related to Hyperparameter Tuning

## Lab

### Load Necessary Libraries

In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

### Load and Prepare Data

We're going to use iris dataset.

In [2]:
# Load dataset
iris = load_iris()
X = iris.data
y = iris.target

# Split into training and testing sets (80-20 split)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


Let's use decision tree again! (Don't forget it yet)

In [3]:
# Initialize a Decision Tree model
dt = DecisionTreeClassifier(random_state=42)

### Parameter Tuning Using GridSearchCV

In [4]:
# Define parameter grid for tuning
param_grid = {
    "criterion": ["gini", "entropy"],
    "max_depth": [3, 5, 10, None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
}

# Apply GridSearchCV
grid_search = GridSearchCV(dt, param_grid, cv=5, scoring="accuracy", n_jobs=-1)
grid_search.fit(X_train, y_train)

# Get best parameters
print("Best Parameters from GridSearchCV:", grid_search.best_params_)


Best Parameters from GridSearchCV: {'criterion': 'entropy', 'max_depth': 5, 'min_samples_leaf': 4, 'min_samples_split': 2}


### Parameter Tuning Using RandomizedSearchCV

In [5]:
from scipy.stats import randint

# Define parameter distributions for tuning
param_dist = {
    "criterion": ["gini", "entropy"],
    "max_depth": [3, 5, 10, None],
    "min_samples_split": randint(2, 20),
    "min_samples_leaf": randint(1, 10),
}

# Apply RandomizedSearchCV
random_search = RandomizedSearchCV(
    dt, param_dist, n_iter=10, cv=5, scoring="accuracy", n_jobs=-1, random_state=42
)
random_search.fit(X_train, y_train)

# Get best parameters
print("Best Parameters from RandomizedSearchCV:", random_search.best_params_)


Best Parameters from RandomizedSearchCV: {'criterion': 'entropy', 'max_depth': None, 'min_samples_leaf': 3, 'min_samples_split': 3}


### Evaluate the Best Model

In [6]:
# Train a Decision Tree using the best parameters from GridSearchCV
best_dt = grid_search.best_estimator_
y_pred = best_dt.predict(X_test)

# Compute accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Test Accuracy with Best GridSearchCV Model: {accuracy:.4f}")


Test Accuracy with Best GridSearchCV Model: 1.0000


___

## Tasks:

Use wine dataset :)

In [7]:
from sklearn.datasets import load_wine

# Load dataset
wine = load_wine()
X = wine.data
y = wine.target

**Task 1: Train-Test Split**

- Split the Wine dataset into training and testing sets using an 80-20 split.
- What is the role of the train-test split in evaluating a machine learning model?

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

wine_dt = DecisionTreeClassifier(random_state=42)

print(f"Training set shape: {X_train.shape}")
print(f"Testing set shape: {X_test.shape}")

Training set shape: (142, 13)
Testing set shape: (36, 13)


The train-test split separates the data into two parts: one for learning patterns and one for checking how well the model performs on unseen data. This helps us measure generalization instead of only measuring how well the model memorizes the training set.

**Task 2: Cross-Validation**

- Implement cross-validation using GridSearchCV and RandomizedSearchCV.
- How does cross-validation help in providing a more reliable estimate of the model's performance?
- Discuss how cross-validation improves the results and prevents overfitting compared to a simple train-test split.


In [9]:
import time
from scipy.stats import randint

param_grid = {
    "criterion": ["gini", "entropy"],
    "max_depth": [3, 5, 10, None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
}

param_dist = {
    "criterion": ["gini", "entropy"],
    "max_depth": [3, 5, 10, None],
    "min_samples_split": randint(2, 20),
    "min_samples_leaf": randint(1, 10),
}

grid_start_time = time.perf_counter()
grid_search_wine = GridSearchCV(
    wine_dt, param_grid, cv=5, scoring="accuracy", n_jobs=-1
)
grid_search_wine.fit(X_train, y_train)
grid_time = time.perf_counter() - grid_start_time

random_start_time = time.perf_counter()
random_search_wine = RandomizedSearchCV(
    wine_dt,
    param_dist,
    n_iter=10,
    cv=5,
    scoring="accuracy",
    n_jobs=-1,
    random_state=42,
)
random_search_wine.fit(X_train, y_train)
random_time = time.perf_counter() - random_start_time

print("GridSearchCV best parameters:", grid_search_wine.best_params_)
print(f"GridSearchCV best CV accuracy: {grid_search_wine.best_score_:.4f}")
print(f"GridSearchCV runtime: {grid_time:.4f} seconds")
print()
print("RandomizedSearchCV best parameters:", random_search_wine.best_params_)
print(f"RandomizedSearchCV best CV accuracy: {random_search_wine.best_score_:.4f}")
print(f"RandomizedSearchCV runtime: {random_time:.4f} seconds")

GridSearchCV best parameters: {'criterion': 'gini', 'max_depth': 3, 'min_samples_leaf': 2, 'min_samples_split': 2}
GridSearchCV best CV accuracy: 0.9303
GridSearchCV runtime: 0.1569 seconds

RandomizedSearchCV best parameters: {'criterion': 'gini', 'max_depth': 10, 'min_samples_leaf': 7, 'min_samples_split': 10}
RandomizedSearchCV best CV accuracy: 0.9227
RandomizedSearchCV runtime: 0.0386 seconds


Cross-validation gives a more reliable estimate because the model is trained and validated on multiple folds of the training set instead of depending on only one split. This reduces the chance that the evaluation is overly optimistic or pessimistic and helps prevent overfitting to a single validation subset.

**Task 3: Hyperparameter Tuning**

- Use GridSearchCV to find the best hyperparameters by exhaustively searching through the parameter grid.
- Use RandomizedSearchCV to sample a fixed number of combinations of hyperparameters.
- Compare the results from both methods in terms of accuracy and computation time.
- Discuss the trade-offs between GridSearchCV and RandomizedSearchCV.

In [10]:
print("Best hyperparameters from GridSearchCV:", grid_search_wine.best_params_)
print(f"GridSearchCV best CV accuracy: {grid_search_wine.best_score_:.4f}")
print(f"GridSearchCV runtime: {grid_time:.4f} seconds")
print()
print("Best hyperparameters from RandomizedSearchCV:", random_search_wine.best_params_)
print(f"RandomizedSearchCV best CV accuracy: {random_search_wine.best_score_:.4f}")
print(f"RandomizedSearchCV runtime: {random_time:.4f} seconds")

Best hyperparameters from GridSearchCV: {'criterion': 'gini', 'max_depth': 3, 'min_samples_leaf': 2, 'min_samples_split': 2}
GridSearchCV best CV accuracy: 0.9303
GridSearchCV runtime: 0.1569 seconds

Best hyperparameters from RandomizedSearchCV: {'criterion': 'gini', 'max_depth': 10, 'min_samples_leaf': 7, 'min_samples_split': 10}
RandomizedSearchCV best CV accuracy: 0.9227
RandomizedSearchCV runtime: 0.0386 seconds


GridSearchCV checks every combination in the parameter grid, so it is more exhaustive but usually takes more time. RandomizedSearchCV tries only a fixed number of sampled combinations, so it is faster and more efficient for larger search spaces, although it may miss the absolute best setting that GridSearchCV could find.

**Task 4: Model Evaluation**

- Using the best model from GridSearchCV or RandomizedSearchCV, make predictions on the test set and calculate the accuracy.
- Compare the performance of the tuned model with a baseline Decision Tree model (without hyperparameter tuning).
- How does hyperparameter tuning affect the accuracy of the Decision Tree model?

In [11]:
baseline_dt = DecisionTreeClassifier(random_state=42)
baseline_dt.fit(X_train, y_train)
baseline_pred = baseline_dt.predict(X_test)
baseline_accuracy = accuracy_score(y_test, baseline_pred)

best_model = grid_search_wine.best_estimator_
best_method = "GridSearchCV"

if random_search_wine.best_score_ > grid_search_wine.best_score_:
    best_model = random_search_wine.best_estimator_
    best_method = "RandomizedSearchCV"

tuned_pred = best_model.predict(X_test)
tuned_accuracy = accuracy_score(y_test, tuned_pred)

print(f"Baseline Decision Tree accuracy: {baseline_accuracy:.4f}")
print(f"Best tuned model ({best_method}) accuracy: {tuned_accuracy:.4f}")

if tuned_accuracy > baseline_accuracy:
    print("The tuned model performs better on the test set.")
elif tuned_accuracy < baseline_accuracy:
    print("The baseline model performs better on the test set.")
else:
    print("Both models perform equally well on the test set.")

Baseline Decision Tree accuracy: 0.9444
Best tuned model (GridSearchCV) accuracy: 1.0000
The tuned model performs better on the test set.


Hyperparameter tuning can improve the Decision Tree by choosing settings that generalize better than the default model. In this notebook, the tuned model should be compared with the baseline accuracy on the test set to see whether the selected hyperparameters lead to better performance on unseen wine samples.